# 🎬 VideoClip Creator IA — TODO EN UNO (Sin API Keys)
### Orquestador + LTX-Video + Análisis Musical + Ensable Todo en Colab GPU T4

**NO NECESITAS NINGUNA API KEY** — Usa Pollinations Text + Pollinations Flux + LTX-Video

**PASO 1:** Runtime → Change runtime type → **GPU T4**  
**PASO 2:** Ejecuta cada celda en orden (Ctrl+F9)  
**PASO 3:** Copia la URL GRANDE que aparece al final y pégala en el Lanzador

In [ ]:
# ⚙️ CELDA 1/5: Instalar TODO (5-8 min)
!pip install -q flask flask-cors fastapi "uvicorn[standard]" nest-asyncio
!pip install -q librosa requests numpy soundfile
!pip install -q diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg pillow
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
print('✅ TODO instalado')

In [ ]:
# 🔐 CELDA 2/5: Configurar token (solo uno necesario)

# Token que el Lanzador usa para autenticarse
# Cópialo exacto del Launcher
VCC_TOKEN = "5a9fba8ba6a04cbbae2a77bd720c0409"

# NO se necesita Gemini, OpenAI, ni ninguna otra API key
# Todo funciona con Pollinations (gratis, sin key)

print('✅ Configuración cargada')
print(f'   VCC_TOKEN: {"OK ✅" if len(VCC_TOKEN) > 10 else "⚠️ PENDIENTE"}')
print(f'   API keys necesarias: NINGUNA 🎉')

In [ ]:
# 🤖 CELDA 3/5: Cargar LTX-Video (3-5 min)
import torch
from diffusers import LTXImageToVideoPipeline

pipe = LTXImageToVideoPipeline.from_pretrained(
    'Lightricks/LTX-Video',
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()
print('✅ LTX-Video listo')

In [ ]:
# ---- IA TEXTUAL: Gemini (primero) → Pollinations (fallback) ----
def ia_texto(prompt: str) -> dict:
    """Intenta Gemini primero; si falla, usa Pollinations. Devuelve JSON."""
    # Intento 1: Gemini
    try:
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent?key={GEMINI_API_KEY}"
        r = requests.post(url, json={
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"response_mime_type": "application/json"},
        }, timeout=120)
        if r.status_code == 200:
            txt = r.json()["candidates"][0]["content"]["parts"][0]["text"]
            return _parse_json(txt)
        print(f"  ⚠️ Gemini {r.status_code}, usando Pollinations...")
    except Exception as e:
        print(f"  ⚠️ Gemini error: {e}, usando Pollinations...")

    # Intento 2: Pollinations Text (gratis, sin key)
    payload = {
        "model": "openai",
        "messages": [
            {"role": "system", "content": "Responde SOLO con JSON valido, sin markdown ni explicaciones."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.9
    }
    r = requests.post("https://text.pollinations.ai/openai", json=payload, timeout=120)
    r.raise_for_status()
    txt = r.json()["choices"][0]["message"]["content"]
    return _parse_json(txt)

def _parse_json(txt: str) -> dict:
    import re
    txt = txt.strip()
    if txt.startswith("```"):
        txt = txt.split("\n", 1)[1] if "\n" in txt else txt[3:]
    if txt.endswith("```"):
        txt = txt[:-3]
    txt = txt.strip()
    try:
        return json.loads(txt)
    except:
        m = re.search(r'\{.*\}', txt, re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise HTTPException(500, f"IA no devolvio JSON: {txt[:200]}")



In [ ]:
# 🌐 CELDA 5/5: Túnel público + URL GRANDE PARA COPIAR
import subprocess, re, time

print('⏳ Creando túnel público con Cloudflare...')
proc = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8080'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

url = None
for _ in range(120):
    line = proc.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break

if not url:
    print('❌ ERROR: No se pudo crear el túnel. Reintenta esta celda.')
else:
    print()
    print('=' * 60)
    print()
    print('    🎬 ¡COLAB LISTO!   Copia esta URL 👇')
    print()
    print(f'    🔗  {url}')
    print()
    print('    PÉGALA en el campo "URL del Orquestador"')
    print('    de la página Lanzador y guarda.')
    print()
    print('    NOTA: No cierres esta pestaña mientras generas.')
    print()
    print('=' * 60)
    print()
    
    # Mantener vivo
    while True:
        time.sleep(60)